# Do Qwen's explanations reflect what changes its answers?

## Phase 2 · Prepare e-SNLI

Freeze 1,000 test examples and select 10 development examples for the later smoke test. This section runs on CPU. On Kaggle, enable internet for the first run. On your computer, use a Python 3.10–3.13 Jupyter kernel.

In [1]:
%pip install -q spacy==3.8.7 vaderSentiment==3.3.2 https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.


### 1. Sample the data and count training words

We use the official [e-SNLI CSVs](https://github.com/OanaMariaCamburu/e-SNLI), keeping their original pair IDs and splits. We deduplicate evaluation pairs by their text after normalizing case and whitespace. The same seed and dataset revision give the same sample. The first run saves the revision in the manifest.

For IDF, one training pair is one document. We count each lowercase word once per pair, keeping apostrophes within words. Unseen words have a document frequency of zero. Target rarity is the mean IDF of its words.

In [2]:
import csv
import json
import math
import random
import re
import sys
from collections import Counter
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from shutil import copyfileobj
from urllib.request import urlopen

SEED = 42
SAMPLE_SIZE = 1000
SMOKE_SIZE = 10
OUTPUT = Path("data")
REPOSITORY = "OanaMariaCamburu/e-SNLI"
SPLITS = {"train": ("train_1", "train_2"), "test": ("test",), "validation": ("dev",)}
LABELS = ("entailment", "neutral", "contradiction")
ROLES = ("premise", "hypothesis")
FAMILIES = ("insertion", "deletion")
WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")
MANIFEST_PATH = OUTPUT / "phase2_manifest.json"

def word_tokens(text):
    return WORD.findall(text.casefold())

def source_revision():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))["dataset_revision"]
    url = f"https://api.github.com/repos/{REPOSITORY}/commits/master"
    with urlopen(url, timeout=30) as response:
        return json.load(response)["sha"]

def split_rows(split, revision):
    cache = OUTPUT / ".cache" / revision
    cache.mkdir(parents=True, exist_ok=True)
    seen = set()
    for part in SPLITS[split]:
        filename = f"esnli_{part}.csv"
        path = cache / filename
        if not path.exists():
            url = f"https://raw.githubusercontent.com/{REPOSITORY}/{revision}/dataset/{filename}"
            temporary = path.with_suffix(".tmp")
            with urlopen(url, timeout=120) as response, temporary.open("wb") as stream:
                copyfileobj(response, stream)
            temporary.replace(path)
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            headers = {name.strip().casefold(): name for name in (reader.fieldnames or [])}
            required = {"pairid", "gold_label", "sentence1", "sentence2"}
            if not required <= headers.keys():
                raise ValueError(f"Unexpected e-SNLI columns in {filename}: {list(headers)}")
            for row in reader:
                source_id, label, premise, hypothesis = (
                    (row[headers[key]] or "").strip()
                    for key in ("pairid", "gold_label", "sentence1", "sentence2")
                )
                if source_id and source_id not in seen and premise and hypothesis and label in LABELS:
                    seen.add(source_id)
                    yield dict(source_id=source_id, split=split, premise=premise,
                               hypothesis=hypothesis, gold_label=label)

def pair_key(row):
    return tuple(" ".join(row[role].casefold().split()) for role in ROLES)

def sample_split(split, size, revision, excluded=()):
    seen = {pair_key(row) for row in excluded}
    unique = []
    for row in sorted(split_rows(split, revision), key=lambda item: item["source_id"]):
        key = pair_key(row)
        if key not in seen:
            seen.add(key)
            unique.append(row)
    return random.Random(f"{SEED}:{split}").sample(unique, size)

def training_word_counts(revision):
    counts = Counter()
    size = 0
    for row in split_rows("train", revision):
        counts.update(set(word_tokens(row["premise"] + "\n" + row["hypothesis"])))
        size += 1
    if not size:
        raise ValueError("The training split is empty.")
    return counts, size

revision = source_revision()
test_examples = sample_split("test", SAMPLE_SIZE, revision)
smoke_examples = sample_split("validation", SMOKE_SIZE, revision, excluded=test_examples)
document_frequency, training_size = training_word_counts(revision)

### 2. Propose and annotate the edits

An insertion adds an adjective, manner adverb, or city. A deletion removes a simple modifier or a trailing adjunct. The seed chooses among eligible sentence roles, then methods, then candidates. When neither sentence has a candidate, the proposal log records the failed attempt.

The character spans are zero-based with an exclusive end. The target span points to the edited sentence for insertion and the original sentence for deletion. Position uses the midpoint of the target's word span in the two-sentence input. spaCy checks for overlapping named entities such as people, organizations, and places, excluding numeric and date labels and VADER scores sentiment on the target alone. Grammatical negation is recorded separately.

In [3]:
from dataclasses import dataclass

import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

NLP = spacy.load("en_core_web_sm")
SENTIMENT = SentimentIntensityAnalyzer()
PEOPLE = set("man woman boy girl child kid person player worker rider skier runner cyclist".split())
OBJECTS = set("shirt jacket coat hat dress sweater pants bicycle bike car truck bag ball umbrella".split())
ACTIONS = set("walk run ride dance play work read write talk speak sing swim jump skate ski paint move".split())
MANNER = ("slowly", "quickly", "quietly", "eagerly", "carefully", "casually", "reluctantly")
CITIES = ("Paris", "Berlin", "Oslo", "Lisbon", "Nairobi", "Accra")
ADJUNCTS = {"in", "at", "on", "near", "under", "beside", "during"}
NAME_TYPES = {"PERSON", "NORP", "FAC", "ORG", "GPE", "LOC", "PRODUCT", "EVENT",
              "WORK_OF_ART", "LAW", "LANGUAGE"}
NEGATION = re.compile(r"\b(?:not|never|no)\b|n['’]t\b", re.IGNORECASE)

@dataclass(frozen=True)
class Edit:
    start: int
    end: int
    replacement: str
    target: str
    method: str

def agrees_with_article(previous, following):
    if previous not in {"a", "an"}:
        return True
    return following.isalpha() and (previous == "an") == (following[0].lower() in "aeiou")

def insertions(doc):
    for token in doc:
        if not token.i or not doc.text[token.idx - 1].isspace():
            continue
        if token.pos_ == "NOUN" and not any(child.dep_ in {"amod", "compound"} for child in token.children):
            adjectives = ()
            if token.lemma_ in PEOPLE:
                adjectives = ("tall", "young", "cheerful", "sad", "anxious", "weary")
            elif token.lemma_ in OBJECTS:
                adjectives = ("red", "blue", "pink", "purple", "crimson", "turquoise")
            for adjective in adjectives:
                if agrees_with_article(doc[token.i - 1].lower_, adjective):
                    yield Edit(token.idx, token.idx, adjective + " ", adjective, "adjective")
        if token.dep_ != "ROOT" or token.pos_ != "VERB" or token.lemma_ not in ACTIONS:
            continue
        if not any(child.dep_ == "advmod" for child in token.children):
            for adverb in MANNER:
                yield Edit(token.idx, token.idx, adverb + " ", adverb, "adverb")
        has_place = any(entity.label_ in {"GPE", "LOC", "FAC"} for entity in doc.ents)
        if not has_place and not any(child.dep_ == "prep" for child in token.children):
            tail = re.search(r"\s*\.?$", doc.text).start()
            if tail and doc.text[tail - 1].isalnum() and not re.search(r"[,;:\"()]", doc.text):
                for city in CITIES:
                    yield Edit(tail, tail, " in " + city, city, "city")

def deletions(doc):
    for token in doc:
        start, end = token.idx, token.idx + len(token)
        if not token.i or not doc.text[start - 1].isspace():
            continue
        simple_adjective = token.dep_ == "amod" and token.pos_ == "ADJ"
        simple_adverb = token.dep_ == "advmod" and token.lower_ in MANNER
        if (simple_adjective or simple_adverb) and not any(token.children) and token.i + 1 < len(doc):
            following = doc[token.i + 1]
            previous = doc[token.i - 1].lower_
            if end < following.idx and WORD.search(token.text) and agrees_with_article(previous, following.text):
                yield Edit(start, following.idx, "", token.text, "modifier")
        if token.dep_ != "prep" or token.lower_ not in ADJUNCTS:
            continue
        if token.head.pos_ != "VERB" or token.head.lemma_ not in ACTIONS:
            continue
        phrase = list(token.subtree)
        if len(phrase) < 2 or phrase[0].i != token.i or phrase[-1].i - token.i + 1 != len(phrase):
            continue
        if any(part.dep_ in {"cc", "conj", "punct"} for part in phrase):
            continue
        phrase_end = phrase[-1].idx + len(phrase[-1])
        if re.fullmatch(r"\s*[.!?]?", doc.text[phrase_end:]):
            target = doc.text[start:phrase_end]
            if word_tokens(target):
                yield Edit(start - 1, phrase_end, "", target, "trailing_adjunct")

def target_features(doc, start, end, target, words_before, total_words):
    entities = sorted({entity.label_ for entity in doc.ents
                       if entity.label_ in NAME_TYPES and entity.start_char < end and entity.end_char > start})
    token_indexes = [i for i, word in enumerate(WORD.finditer(doc.text))
                     if word.start() < end and word.end() > start]
    position = (words_before + (token_indexes[0] + token_indexes[-1] + 1) / 2) / total_words
    score = SENTIMENT.polarity_scores(target)["compound"]
    polarity = "negative" if score <= -0.05 else "positive" if score >= 0.05 else "neutral"
    idfs = [math.log((training_size + 1) / (document_frequency[word] + 1)) + 1
            for word in word_tokens(target)]
    return dict(named_entity=int(bool(entities)), entity_types="|".join(entities),
                vader_score=score, polarity=polarity, negation_in_target=int(bool(NEGATION.search(target))),
                position=position, position_bin=("early", "middle", "late")[min(2, int(position * 3))],
                idf=sum(idfs) / len(idfs))

def make_proposal(row, docs, family):
    insertion = family == "insertion"
    proposal = {**row, "edit_id": f"{row['source_id']}:{family}", "family": family,
                "edited_premise": row["premise"], "edited_hypothesis": row["hypothesis"]}
    original_words = set(word_tokens(row["premise"] + "\n" + row["hypothesis"]))
    edits_by_role = {}
    for role, doc in docs.items():
        edits = list(insertions(doc) if insertion else deletions(doc))
        if insertion:
            edits = [edit for edit in edits if original_words.isdisjoint(word_tokens(edit.target))]
        if edits:
            edits_by_role[role] = edits
    if not edits_by_role:
        return {**proposal, "status": "rejected", "rejection_reason": "no eligible rule-based edit"}

    rng = random.Random(f"{SEED}:{proposal['edit_id']}")
    role = rng.choice(list(edits_by_role))
    edits = edits_by_role[role]
    method = rng.choice(sorted({edit.method for edit in edits}))
    edit = rng.choice([edit for edit in edits if edit.method == method])
    original = row[role]
    edited = original[:edit.start] + edit.replacement + original[edit.end:]
    span_doc = NLP(edited) if insertion else docs[role]
    changed_text = edit.replacement if insertion else original[edit.start:edit.end]
    start = edit.start + changed_text.index(edit.target)
    end = start + len(edit.target)
    assert span_doc.text[start:end] == edit.target
    proposal[f"edited_{role}"] = edited
    reference = {key: proposal[f"edited_{key}"] if insertion else row[key] for key in ROLES}
    premise_words = len(word_tokens(reference["premise"]))
    total_words = premise_words + len(word_tokens(reference["hypothesis"]))
    features = target_features(span_doc, start, end, edit.target,
                               premise_words if role == "hypothesis" else 0, total_words)
    return {**proposal, **features, "status": "proposed", "method": method, "role": role,
            "target": edit.target, "target_start": start, "target_end": end,
            "span_text": "edited" if insertion else "original", "patch_start": edit.start,
            "patch_end": edit.end, "replacement": edit.replacement}

def propose_edits(examples):
    texts = (row[role] for row in examples for role in ROLES)
    documents = iter(NLP.pipe(texts, batch_size=64))
    proposals = []
    for row in examples:
        docs = {role: next(documents) for role in ROLES}
        proposals.extend(make_proposal(row, docs, family) for family in FAMILIES)
    return proposals

test_edits = propose_edits(test_examples)
smoke_edits = propose_edits(smoke_examples)

### 3. Freeze the proposals and review 20 edits

We divide the test edit IDF scores into three groups within each edit family using the nearest-rank 33rd and 67th percentiles. Equal scores stay together. The same cutoffs are used for the development examples.

Review ten insertions and ten deletions in `data/edit_audit.csv`. Compare each original and edited pair. Record whether it reads naturally, makes sense, changes the meaning, might change the correct label, and has a correct entity annotation. Use `accept` or `reject` in `review_decision` and explain any rejection in `review_notes`. The audit deliberately includes up to three named-entity edits per family. It checks edit quality and annotation errors and is separate from the model evaluation.

In [4]:
def assign_rarity_and_select_audit():
    thresholds = {}
    audit = []
    for family in FAMILIES:
        proposed = [row for row in test_edits if row["family"] == family and row["status"] == "proposed"]
        scores = sorted(row["idf"] for row in proposed)
        if not scores:
            thresholds[family] = None
            continue
        low, high = (scores[math.ceil(len(scores) * q) - 1] for q in (1 / 3, 2 / 3))
        thresholds[family] = [low, high]
        for row in test_edits + smoke_edits:
            if row["family"] == family and row["status"] == "proposed":
                row["rarity_bin"] = "common" if row["idf"] <= low else "middle" if row["idf"] <= high else "rare"

        rng = random.Random(f"{SEED}:audit:{family}")
        entities = [row for row in proposed if row["named_entity"]]
        picked = rng.sample(entities, min(3, len(entities)))
        picked_ids = {row["edit_id"] for row in picked}
        other = [row for row in proposed if row["edit_id"] not in picked_ids]
        picked.extend(rng.sample(other, min(10 - len(picked), len(other))))
        audit_columns = ("edit_id", "family", "role", "target", "premise", "hypothesis",
                         "edited_premise", "edited_hypothesis", "gold_label", "entity_types")
        audit.extend({**{key: row[key] for key in audit_columns},
                      "grammatical": "", "coherent": "", "meaning_changed": "",
                      "gold_label_may_change": "", "entity_annotation_correct": "",
                      "review_decision": "", "review_notes": ""} for row in picked)
    return thresholds, audit

def export_csv(path, rows, expected_hash=None):
    fields = list(dict.fromkeys(field for row in rows for field in row))
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    digest = sha256(temporary.read_bytes()).hexdigest()
    if expected_hash is not None and digest != expected_hash:
        temporary.unlink()
        raise ValueError(f"Regenerated data differs from the frozen version of {path.name}.")
    temporary.replace(path)
    return digest

thresholds, audit = assign_rarity_and_select_audit()
files = {"frozen_examples.csv": test_examples, "edit_proposals.csv": test_edits,
         "smoke_examples.csv": smoke_examples, "smoke_edits.csv": smoke_edits}
OUTPUT.mkdir(exist_ok=True)
previous = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else None
if previous:
    if (previous["seed"], previous["test_size"], previous["smoke_size"]) != (SEED, SAMPLE_SIZE, SMOKE_SIZE):
        raise ValueError("The output directory already contains a sample frozen with different settings.")
hashes = {name: export_csv(OUTPUT / name, rows, previous["file_sha256"][name] if previous else None)
          for name, rows in files.items()}
audit_path = OUTPUT / "edit_audit.csv"
if not audit_path.exists() and audit:
    export_csv(audit_path, audit)

counts = Counter(row["gold_label"] for row in test_examples)
proposals = Counter(f"{row['family']}_{row['status']}" for row in test_edits)
edits_per_example = Counter(row["source_id"] for row in test_edits if row["status"] == "proposed")
coverage = Counter(edits_per_example[row["source_id"]] for row in test_examples)
manifest = {
    "dataset_source": f"https://github.com/{REPOSITORY}",
    "dataset_revision": revision, "seed": SEED,
    "test_split": "test", "smoke_split": "validation",
    "test_size": len(test_examples), "smoke_size": len(smoke_examples),
    "training_documents": training_size,
    "label_counts": dict(counts),
    "label_proportions": {label: counts[label] / len(test_examples) for label in LABELS},
    "proposal_counts": dict(proposals), "idf_thresholds": thresholds,
    "examples_by_proposal_count": {str(count): coverage[count] for count in range(3)},
    "versions": {"python": sys.version.split()[0], **{
        name: version(name) for name in ("spacy", "en-core-web-sm", "vaderSentiment")
    }},
    "file_sha256": hashes,
}
if previous is None:
    temporary = MANIFEST_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    temporary.replace(MANIFEST_PATH)

print(f"Frozen test examples: {len(test_examples)}. Validation smoke examples: {len(smoke_examples)}.")
print("Test label counts:", dict(counts))
print("Edit proposal counts:", dict(proposals))
print("Examples with zero, one, or two proposals:", [coverage[count] for count in range(3)])
print(f"Audit rows available: {len(audit)}. Review at least 20 before running the main evaluation.")
if len(audit) < 20:
    print("The audit needs more eligible edits.")

Frozen test examples: 1000. Validation smoke examples: 10.
Test label counts: {'contradiction': 343, 'neutral': 294, 'entailment': 363}
Edit proposal counts: {'insertion_rejected': 252, 'deletion_proposed': 664, 'deletion_rejected': 336, 'insertion_proposed': 748}
Examples with zero, one, or two proposals: [104, 380, 516]
Audit rows available: 20. Review at least 20 before running the main evaluation.
